# Control: does per-subject normalisation rescue node-variance LOSO transfer?

Same LOSO pipeline as `Figure_3_Neuropixels.ipynb` (cell 8), but node-variance features are normalised **within each subject** before pooling. Variants: `raw` (original), `z` (per-column z-score), `logz` (log, then per-column z-score). Correlation edges are included as reference.

In [1]:
import os, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))
from src.neuropixels_data import list_subjects, load_subject_lfp
from src.features import build_feature
from src.util import moving_average_centered
from src.sfa import loso_cv

os.environ["NEUROPIXELS_DATA_ROOT"] = "/Users/ritviksharma/Projects/code_pSFA/experiments/neuropixels/functional_connectivity"
DATA_ROOT = Path(os.environ["NEUROPIXELS_DATA_ROOT"])
CACHE_DIR = Path("outputs/results/neuropixels"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
DOWNSAMPLE_FACTOR=10; EDGE_WIN=1501; N_COMPONENTS=3
POOL_AREAS=["VISp","VISrl","VISal","VISam"]
CORRUPTED_SIDS=["786091066","829720705","831882777","835479236","839068429"]
def drop_corrupted(df): return df[~df.index.astype(str).isin(CORRUPTED_SIDS)]
subjects = list_subjects(DATA_ROOT)

In [2]:
def zcol(F):
    return (F - F.mean(0, keepdims=True)) / F.std(0, keepdims=True)

TRANSFORMS = {
    "raw":  lambda F: F,
    "z":    zcol,
    "logz": lambda F: zcol(np.log(F)),
}

pool = {}
for sid in subjects:
    out = load_subject_lfp(DATA_ROOT/sid, DOWNSAMPLE_FACTOR, area_filter=POOL_AREAS, strict=True)
    if out is not None: pool[sid] = out

F_corr, F_var, d = {}, {}, {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for sid,(X,d_raw,_,_) in pool.items():
        Fc = build_feature("edges-corr", X, EDGE_WIN)
        Fv = build_feature("node-var", X, EDGE_WIN)
        dd = moving_average_centered(d_raw[:,None], EDGE_WIN)[:,0]
        n = min(len(dd), Fc.shape[0], Fv.shape[0])
        F_corr[sid]=Fc[:n]; F_var[sid]=Fv[:n]; d[sid]=dd[:n]
print(len(pool), "eligible subjects; min node variance across subjects:", min(F.min() for F in F_var.values()))

13 eligible subjects; min node variance across subjects: 4.705067762117318e-10


In [3]:
cols = {"corr (reference)": loso_cv(F_corr, d, N_COMPONENTS)}
for name, f in TRANSFORMS.items():
    Fd = {s: f(F) for s,F in F_var.items()}
    cols[f"nodevar_{name}"] = loso_cv(Fd, d, N_COMPONENTS)
res = pd.DataFrame(cols)
res.to_csv(CACHE_DIR/"loso_nodevar_normalisation_control.csv")
g = drop_corrupted(res)
print(f"N = {len(g)} folds after QC exclusion")
display(g.round(3))
display(g.agg(["median","mean"]).round(3))

N = 11 folds after QC exclusion


,corr (reference),nodevar_raw,nodevar_z,nodevar_logz
766640955,0.509,0.004,0.380,0.461
767871931,0.733,0.004,0.577,0.623
771990200,0.833,0.556,0.590,0.633
774875821,0.679,0.116,0.509,0.644
778240327,0.691,0.096,0.668,0.682
778998620,0.778,0.166,0.588,0.710
781842082,0.844,0.251,0.624,0.805
787025148,0.639,0.000,0.395,0.446
789848216,0.874,0.002,0.535,0.552
794812542,0.689,0.339,0.634,0.786


,corr (reference),nodevar_raw,nodevar_z,nodevar_logz
median,0.691,0.116,0.577,0.633
mean,0.701,0.161,0.516,0.593


In [4]:
# per-subject scale heterogeneity of raw node variance (motivation for the control)
sc = pd.DataFrame({s: F.mean(0) for s,F in F_var.items()}, index=POOL_AREAS).T
sc = drop_corrupted(sc)
display(sc.round(1))
print("max/min mean-variance ratio across subjects, per area:"); print((sc.max()/sc.min()).round(1))

,VISp,VISrl,VISal,VISam
766640955,0.0,0.0,0.0,0.0
767871931,0.0,0.0,0.0,0.0
771990200,0.0,0.0,0.0,0.0
774875821,0.0,0.0,0.0,0.0
778240327,0.0,0.0,0.0,0.0
778998620,0.0,0.0,0.0,0.0
781842082,0.0,0.0,0.0,0.0
787025148,0.0,0.0,0.0,0.0
789848216,0.0,0.0,0.0,0.0
794812542,0.0,0.0,0.0,0.0


max/min mean-variance ratio across subjects, per area:
VISp      7.3
VISrl     7.5
VISal    13.1
VISam     3.5
dtype: float64
